In [5]:
!pip install tensorflow scikit-learn vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.5 MB/s eta 0:00:00


In [6]:
import pandas as pd, numpy as np, re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---- তোমার দুটো file এর নাম এখানে (upload করার পর মিলিয়ে নাও) ----
NEWS_CSV = "nepal_flood_news_v2.csv"          # newspaper
NEWS_TEXT = "full_text"
YT_CSV   = "nepal_youtube_comments_clean.csv"  # youtube
YT_TEXT  = "comment_text"
# ----------------------------------------------------------------

def clean(t):
    t = re.sub(r"http\S+"," ",str(t)); t = re.sub(r"<[^>]+>"," ",t)
    return re.sub(r"\s+"," ",t).strip()

def to_sentences(df, tcol, source):
    rows=[]
    for _,r in df.iterrows():
        for s in re.split(r"(?<=[.!?])\s+", clean(r[tcol])):
            if len(s.split())>=5:
                rows.append({"source":source, "text":s.strip()})
    return pd.DataFrame(rows)

news = pd.read_csv(NEWS_CSV); yt = pd.read_csv(YT_CSV)
sents = pd.concat([to_sentences(news,NEWS_TEXT,"newspaper"),
                   to_sentences(yt,YT_TEXT,"youtube")], ignore_index=True)
print("মোট sentence:", len(sents))

# VADER sentiment label (hybrid এর target)
vader = SentimentIntensityAnalyzer()
def vlab(t):
    c = vader.polarity_scores(str(t))["compound"]
    return "positive" if c>=0.05 else "negative" if c<=-0.05 else "neutral"
sents["vader"] = sents["text"].apply(vlab)
print(sents["vader"].value_counts())

মোট sentence: 25646
vader
negative    10812
positive     8026
neutral      6808
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = sents.dropna(subset=["vader"]).copy()
df["text"] = df["text"].astype(str)
le = LabelEncoder(); df["y"] = le.fit_transform(df["vader"])
print("Classes:", list(le.classes_))

X_train, X_test, y_train, y_test = train_test_split(
    df["text"].values, df["y"].values, test_size=0.25, random_state=42, stratify=df["y"])

MAX_WORDS, MAX_LEN = 20000, 60
tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(X_train)
Xtr = pad_sequences(tok.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post", truncating="post")
Xte = pad_sequences(tok.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding="post", truncating="post")
print("Train:", Xtr.shape, "Test:", Xte.shape)

Classes: ['negative', 'neutral', 'positive']
Train: (19234, 60) Test: (6412, 60)


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Conv1D, MaxPooling1D,
    Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D, concatenate)
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

n_classes = len(le.classes_)
inp = Input(shape=(MAX_LEN,))
emb = Dropout(0.3)(Embedding(MAX_WORDS, 128, input_length=MAX_LEN)(inp))

# CNN branch
cnn = Conv1D(128,5,activation="relu",padding="same")(emb)
cnn = MaxPooling1D(2)(cnn)
cnn = Conv1D(64,3,activation="relu",padding="same")(cnn)
cnn = GlobalMaxPooling1D()(cnn)

# BiLSTM branch
lstm = Bidirectional(LSTM(64,return_sequences=True))(emb)
lstm = Bidirectional(LSTM(32))(lstm)

merged = concatenate([cnn, lstm])
x = Dropout(0.4)(Dense(64,activation="relu")(merged))
out = Dense(n_classes,activation="softmax")(x)

model = Model(inp,out)
model.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

cw = dict(enumerate(compute_class_weight("balanced",classes=np.unique(y_train),y=y_train)))
history = model.fit(Xtr,y_train,validation_split=0.1,epochs=10,batch_size=128,class_weight=cw,verbose=1)

Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


136/136 ━━━━━━━━━━━━━━━━━━━━ 16s 57ms/step - accuracy: 0.6037 - loss: 0.8219 - val_accuracy: 0.8425 - val_loss: 0.4390
Epoch 2/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.8956 - loss: 0.3158 - val_accuracy: 0.8966 - val_loss: 0.3001
Epoch 3/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9519 - loss: 0.1628 - val_accuracy: 0.8960 - val_loss: 0.3106
Epoch 4/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9695 - loss: 0.1023 - val_accuracy: 0.8945 - val_loss: 0.3543
Epoch 5/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9806 - loss: 0.0676 - val_accuracy: 0.8935 - val_loss: 0.4092
Epoch 6/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9854 - loss: 0.0487 - val_accuracy: 0.8971 - val_loss: 0.4322
Epoch 7/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9906 - loss: 0.0338 - val_accuracy: 0.8919 - val_loss: 0.4856
Epoch 8/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9931 - loss: 0.0266 - val_accuracy: 0.88

In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_pred = model.predict(Xte,batch_size=128).argmax(axis=1)
h_acc = accuracy_score(y_test,y_pred); h_f1 = f1_score(y_test,y_pred,average="macro")
print(f"\nHYBRID (CNN+BiLSTM): accuracy={h_acc:.3f}, macro-F1={h_f1:.3f}\n")
print(classification_report(y_test,y_pred,target_names=le.classes_))

comparison = pd.DataFrame({
    "Model":["Logistic Regression","Linear SVM","Random Forest","XGBoost","CNN+BiLSTM (Hybrid)"],
    "Accuracy":[0.863,0.877,0.877,0.872,round(h_acc,3)],
    "Macro-F1":[0.860,0.874,0.874,0.869,round(h_f1,3)]
})
print("\n=== সব model তুলনা ===")
print(comparison.to_string(index=False))
comparison.to_csv("model_comparison_with_hybrid.csv",index=False)

51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step

HYBRID (CNN+BiLSTM): accuracy=0.884, macro-F1=0.881

              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      2703
     neutral       0.89      0.85      0.87      1702
    positive       0.85      0.90      0.87      2007

    accuracy                           0.88      6412
   macro avg       0.88      0.88      0.88      6412
weighted avg       0.88      0.88      0.88      6412


=== সব model তুলনা ===
              Model  Accuracy  Macro-F1
Logistic Regression     0.863     0.860
         Linear SVM     0.877     0.874
      Random Forest     0.877     0.874
            XGBoost     0.872     0.869
CNN+BiLSTM (Hybrid)     0.884     0.881


In [10]:
!pip install tensorflow scikit-learn -q
import pandas as pd, numpy as np
res = pd.read_csv("results_with_docid.csv")   # আমার দেওয়া ফাইল upload করো
for c in ["source","text","vader","doc_id"]: res[c]=res[c].astype(str)
print(len(res), "sentences |", res["doc_id"].nunique(), "documents")

25646 sentences | 3890 documents


In [11]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

le = LabelEncoder(); y = le.fit_transform(res["vader"].values)
X = res["text"].values; groups = res["doc_id"].values

# একই document এর sentence শুধু train বা শুধু test এ যাবে
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(X, y, groups))
print("Train:", len(tr), "Test:", len(te))
# verify no leakage:
print("Overlap docs (should be 0):", len(set(groups[tr]) & set(groups[te])))

MAX_WORDS, MAX_LEN = 20000, 60
tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>"); tok.fit_on_texts(X[tr])
Xtr = pad_sequences(tok.texts_to_sequences(X[tr]), maxlen=MAX_LEN, padding="post", truncating="post")
Xte = pad_sequences(tok.texts_to_sequences(X[te]), maxlen=MAX_LEN, padding="post", truncating="post")
ytr, yte = y[tr], y[te]

Train: 15739 Test: 9907
Overlap docs (should be 0): 0


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Conv1D, MaxPooling1D,
    Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D, concatenate)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

n = len(le.classes_)
cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

def embed(inp): return Dropout(0.3)(Embedding(MAX_WORDS,128)(inp))

def build(kind):
    inp = Input(shape=(MAX_LEN,)); e = embed(inp)
    if kind=="cnn":
        x = Conv1D(128,5,activation="relu",padding="same")(e); x=MaxPooling1D(2)(x)
        x = Conv1D(64,3,activation="relu",padding="same")(x); x=GlobalMaxPooling1D()(x)
    elif kind=="bilstm":
        x = Bidirectional(LSTM(64,return_sequences=True))(e); x=Bidirectional(LSTM(32))(x)
    else:  # hybrid
        cnn = Conv1D(128,5,activation="relu",padding="same")(e); cnn=MaxPooling1D(2)(cnn)
        cnn = Conv1D(64,3,activation="relu",padding="same")(cnn); cnn=GlobalMaxPooling1D()(cnn)
        lstm = Bidirectional(LSTM(64,return_sequences=True))(e); lstm=Bidirectional(LSTM(32))(lstm)
        x = concatenate([cnn,lstm])
    x = Dropout(0.4)(Dense(64,activation="relu")(x))
    out = Dense(n,activation="softmax")(x)
    m = Model(inp,out); m.compile(loss="sparse_categorical_crossentropy",optimizer="adam",metrics=["accuracy"])
    return m

results={}
for kind in ["cnn","bilstm","hybrid"]:
    print(f"\n=== Training {kind} ===")
    m = build(kind)
    m.fit(Xtr,ytr,validation_split=0.1,epochs=6,batch_size=128,class_weight=cw,verbose=1)
    pred = m.predict(Xte,batch_size=128).argmax(1)
    acc=accuracy_score(yte,pred); f1=f1_score(yte,pred,average="macro")
    results[kind]={"acc":acc,"f1":f1}
    print(f"{kind}: acc={acc:.3f}, macro-F1={f1:.3f}")
    if kind=="hybrid":
        print("Confusion matrix:\n", confusion_matrix(yte,pred))
        print(classification_report(yte,pred,target_names=le.classes_))

print("\n=== ABLATION (group-aware, leakage-free) ===")
for k,v in results.items(): print(f'{k:10s}: acc={v["acc"]:.3f}, macro-F1={v["f1"]:.3f}')


=== Training cnn ===
Epoch 1/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 18s 104ms/step - accuracy: 0.5835 - loss: 0.8775 - val_accuracy: 0.6741 - val_loss: 0.7713
Epoch 2/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8719 - loss: 0.3807 - val_accuracy: 0.7643 - val_loss: 0.6014
Epoch 3/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9391 - loss: 0.2038 - val_accuracy: 0.7713 - val_loss: 0.6334
Epoch 4/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9650 - loss: 0.1231 - val_accuracy: 0.7643 - val_loss: 0.6921
Epoch 5/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9771 - loss: 0.0817 - val_accuracy: 0.7675 - val_loss: 0.8251
Epoch 6/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9826 - loss: 0.0630 - val_accuracy: 0.7694 - val_loss: 0.9404
78/78 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
cnn: acc=0.814, macro-F1=0.809

=== Training bilstm ===
Epoch 1/6
111/111 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.6018 - loss: 0.8367 - val_accuracy: 0.6455 - val_los

In [14]:
# এটা তখনই চলবে যদি আপনার DataFrame-এ দুটো GoEmotions label (goemotion + modernbert_emotion) থাকে
from sklearn.metrics import cohen_kappa_score

# 'sents' এর বদলে আপলোড করা 'res' বা অন্য কোনো ফাইলে এই কলামগুলো আছে কি না তা চেক করা হচ্ছে
target_df = res if 'res' in globals() else sents

try:
    k = cohen_kappa_score(target_df["goemotion"], target_df["modernbert_emotion"])
    agree = (target_df["goemotion"] == target_df["modernbert_emotion"]).mean()
    print(f"RoBERTa vs ModernBERT: agreement={agree:.1%}, Cohen's kappa={k:.3f}")
    print("(kappa >0.6 = substantial, >0.8 = almost perfect)")
except KeyError as e:
    print(f"ভুল: {e} কলামটি ডেটাসেটে খুঁজে পাওয়া যায়নি।")
    print("বর্তমান কলামসমূহ:", list(target_df.columns))
    print("ModernBERT এবং RoBERTa দিয়ে আগে প্রেডিকশন রান করুন, অথবা সঠিক ফাইলটি লোড করুন।")
except NameError:
    print("ডেটাসেট খুঁজে পাওয়া যায়নি। দয়া করে আগের কোড সেলগুলো রান করুন।")

ভুল: 'modernbert_emotion' কলামটি ডেটাসেটে খুঁজে পাওয়া যায়নি।
বর্তমান কলামসমূহ: ['source', 'date', 'text', 'nrc_anger', 'nrc_fear', 'nrc_joy', 'nrc_sadness', 'nrc_trust', 'nrc_disgust', 'nrc_surprise', 'nrc_anticipation', 'vader', 'textblob', 'roberta_sent', 'goemotion', 'doc_id', 'week']
ModernBERT এবং RoBERTa দিয়ে আগে প্রেডিকশন রান করুন, অথবা সঠিক ফাইলটি লোড করুন।


In [15]:
from sklearn.metrics import cohen_kappa_score
k = cohen_kappa_score(sents["goemotion"], sents["modernbert_emotion"])
agree = (sents["goemotion"]==sents["modernbert_emotion"]).mean()
print(f"agreement = {agree:.1%}")
print(f"Cohen's kappa = {k:.3f}")
print("(>0.6 substantial, >0.8 almost perfect)")

KeyError: 'goemotion'